# 🎯 Reranking Models

**Day 8 — Notebook 3 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Why retrieval alone isn't enough — the precision problem
- How cross-encoder reranking works vs bi-encoder retrieval
- Implementing LLM-based reranking with Gemini
- Score normalisation and threshold filtering
- Building a complete retrieve-then-rerank pipeline

---

## The Two-Stage Retrieval Pattern

```
Query
  │
  ▼
Stage 1 — Fast retrieval (bi-encoder)
  • Embed query independently
  • ANN search over millions of docs
  • Returns top-K candidates (e.g. K=50)
  │
  ▼
Stage 2 — Precise reranking (cross-encoder)
  • Query + candidate seen TOGETHER
  • Scores each pair more accurately
  • Returns top-N (e.g. N=5)
  │
  ▼
LLM Generation
```

**Why two stages?**  
Cross-encoders are accurate but slow — they must see every (query, doc) pair.  
Bi-encoders are fast but less precise — they embed query and docs independently.  
The combo gets you both speed and precision.

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" chromadb

In [ ]:
import sys, os, json
import chromadb

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from pydantic import BaseModel
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
GEN_MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Precision Problem

Dense retrieval returns the *most similar* chunks — but similarity isn't the same as relevance.  
Let's build a corpus that exposes this gap.

In [ ]:
# Scientific abstract corpus — chosen to expose retrieval precision issues
ABSTRACTS = [
    {
        "id": "0",
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "text": "We introduce BERT, which stands for Bidirectional Encoder Representations "
                "from Transformers. BERT is designed to pre-train deep bidirectional "
                "representations from unlabeled text by jointly conditioning on both left "
                "and right context in all layers. The pre-trained BERT model can be "
                "fine-tuned with just one additional output layer to create state-of-the-art "
                "models for a wide range of tasks, such as question answering and language "
                "inference, without substantial task-specific architecture modifications.",
    },
    {
        "id": "1",
        "title": "Attention Is All You Need",
        "text": "The dominant sequence transduction models are based on complex recurrent or "
                "convolutional neural networks that include an encoder and a decoder. The best "
                "performing models also connect the encoder and decoder through an attention "
                "mechanism. We propose a new simple network architecture, the Transformer, "
                "based solely on attention mechanisms, dispensing with recurrence and convolutions "
                "entirely. Experiments on two machine translation tasks show these models to be "
                "superior in quality while being more parallelizable and requiring significantly "
                "less time to train.",
    },
    {
        "id": "2",
        "title": "GPT-3: Language Models are Few-Shot Learners",
        "text": "We train GPT-3, an autoregressive language model with 175 billion parameters, "
                "and test its performance in the few-shot setting. GPT-3 achieves strong "
                "performance on many NLP datasets, including translation, question-answering, "
                "and cloze tasks, as well as several tasks that require on-the-fly reasoning "
                "or domain adaptation, such as unscrambling words, using novel words in a "
                "sentence, or performing 3-digit arithmetic.",
    },
    {
        "id": "3",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "text": "Large pre-trained language models have been shown to store factual knowledge "
                "implicitly in their parameters, and achieve state-of-the-art results when "
                "fine-tuned on downstream NLP tasks. However, their ability to access and "
                "precisely manipulate knowledge is still limited, and hence on knowledge-intensive "
                "tasks, their performance lags behind task-specific architectures. We explore a "
                "general-purpose fine-tuning recipe for retrieval-augmented generation (RAG) — "
                "models that combine pre-trained parametric and non-parametric memory for "
                "language generation.",
    },
    {
        "id": "4",
        "title": "Dense Passage Retrieval for Open-Domain Question Answering",
        "text": "Open-domain question answering relies on efficient passage retrieval to select "
                "candidate contexts, where traditional sparse vector space models, such as TF-IDF "
                "or BM25, are the de facto method. In this work, we show that retrieval can be "
                "practically implemented using dense representations alone, where embeddings are "
                "learned from a small number of questions and passages by a simple dual-encoder "
                "framework.",
    },
    {
        "id": "5",
        "title": "Improving Language Understanding by Generative Pre-Training",
        "text": "Natural language understanding comprises a wide range of diverse tasks such as "
                "textual entailment, question answering, semantic similarity assessment, and "
                "document classification. Although large unlabeled text corpora are abundant, "
                "labeled data for learning these specific tasks is scarce. We demonstrate that "
                "large gains on these tasks can be realized by generative pre-training of a "
                "language model on a diverse corpus of unlabeled text, followed by discriminative "
                "fine-tuning on each specific task.",
    },
    {
        "id": "6",
        "title": "Scaling Laws for Neural Language Models",
        "text": "We study empirical scaling laws for language model performance on the cross-entropy "
                "loss. The loss scales as a power-law with model size, dataset size, and the "
                "amount of compute used for training, with some trends spanning more than seven "
                "orders of magnitude. Other architectural details such as network width or depth "
                "have minimal effects within a wide range.",
    },
    {
        "id": "7",
        "title": "LoRA: Low-Rank Adaptation of Large Language Models",
        "text": "An important paradigm of natural language processing consists of large-scale "
                "pre-training on general domain data and adaptation to particular tasks or domains. "
                "We propose Low-Rank Adaptation (LoRA), which freezes the pre-trained model "
                "weights and injects trainable rank decomposition matrices into each layer of the "
                "Transformer architecture, greatly reducing the number of trainable parameters for "
                "downstream tasks.",
    },
    {
        "id": "8",
        "title": "Chain-of-Thought Prompting Elicits Reasoning in Large Language Models",
        "text": "We explore how generating a chain of thought — a series of intermediate reasoning "
                "steps — significantly improves the ability of large language models to perform "
                "complex reasoning. A striking empirical finding is that chain-of-thought prompting "
                "improves performance on a range of arithmetic, commonsense, and symbolic "
                "reasoning tasks. The gains from chain-of-thought prompting are larger for "
                "bigger models.",
    },
    {
        "id": "9",
        "title": "Efficient Transformers: A Survey",
        "text": "Transformers have become the de facto standard for many natural language processing "
                "tasks. However, self-attention in standard Transformers has quadratic time and "
                "memory complexity with respect to input sequence length, which limits their "
                "applicability. This survey covers efficient Transformer models, discussing various "
                "approaches for reducing the quadratic complexity, including sparse attention, "
                "linear approximations, and memory-augmented architectures.",
    },
]

print(f"Corpus: {len(ABSTRACTS)} abstracts")
for a in ABSTRACTS:
    print(f"  [{a['id']}] {a['title']}")

In [ ]:
# Build ChromaDB index
chroma = chromadb.Client()
col = chroma.create_collection("abstracts", metadata={"hnsw:space": "cosine"})

texts = [a["text"] for a in ABSTRACTS]
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=texts,
    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
)
col.add(
    ids=[a["id"] for a in ABSTRACTS],
    documents=texts,
    embeddings=[e.values for e in result.embeddings],
    metadatas=[{"title": a["title"]} for a in ABSTRACTS],
)
print(f"✅ Indexed {col.count()} documents")

In [ ]:
def dense_retrieve(query: str, top_k: int = 5) -> list[dict]:
    """Standard dense retrieval — returns top_k candidates."""
    q_vec = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values

    results = col.query(
        query_embeddings=[q_vec],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "id": results["ids"][0][i],
            "title": results["metadatas"][0][i]["title"],
            "text": results["documents"][0][i],
            "dense_score": 1 - results["distances"][0][i],  # cosine similarity
        }
        for i in range(len(results["ids"][0]))
    ]


# Test retrieval — notice what comes back for a specific question
query = "How do you adapt a large pre-trained model without retraining all weights?"
candidates = dense_retrieve(query, top_k=5)

print(f"Query: '{query}'\n")
print("Dense retrieval top-5:")
for i, c in enumerate(candidates):
    print(f"  {i+1}. [{c['id']}] {c['title']}")
    print(f"     dense_score={c['dense_score']:.4f}")

print("\n💡 LoRA (id=7) is the most relevant — did dense retrieval rank it #1?")

---

## 2. LLM-Based Reranking

Without access to a dedicated cross-encoder like Cohere Rerank, we can use Gemini  
itself to score (query, document) pairs. This is slower but surprisingly effective.

In [ ]:
class RelevanceScore(BaseModel):
    score: int          # 1–5
    reasoning: str      # brief justification


def llm_score_pair(query: str, document: str) -> RelevanceScore:
    """Ask Gemini to score how relevant a document is for a query."""
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Score the relevance of the following document to the query.

Query: {query}

Document: {document}

Score from 1 to 5:
5 = Directly and completely answers the query
4 = Highly relevant, covers the main topic
3 = Partially relevant, touches the topic
2 = Tangentially related
1 = Not relevant""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RelevanceScore,
            temperature=0.0,
        ),
    )
    return RelevanceScore.model_validate_json(response.text)


def llm_rerank(query: str, candidates: list[dict]) -> list[dict]:
    """Score each candidate and return sorted by LLM relevance score."""
    reranked = []
    for c in candidates:
        scored = llm_score_pair(query, c["text"])
        reranked.append({
            **c,
            "llm_score": scored.score,
            "reasoning": scored.reasoning,
        })
    return sorted(reranked, key=lambda x: x["llm_score"], reverse=True)


print("Testing LLM reranker on the same query...")
reranked = llm_rerank(query, candidates)

print(f"\nQuery: '{query}'\n")
print(f"{'Rank':<6} {'LLM':>5} {'Dense':>7}  Title")
print("-" * 70)
for i, c in enumerate(reranked):
    print(f"  {i+1}    {c['llm_score']}/5   {c['dense_score']:.4f}  {c['title']}")
    print(f"         Reason: {c['reasoning'][:80]}")

---

## 3. Pointwise vs Listwise Reranking

The approach above is **pointwise** — each document is scored independently.  
**Listwise** reranking shows all candidates at once and asks for a ranking — it's more efficient.

In [ ]:
class RankedList(BaseModel):
    ranked_ids: list[str]   # document IDs in order of relevance (most → least)
    reasoning: str          # brief overall explanation


def listwise_rerank(query: str, candidates: list[dict]) -> list[dict]:
    """Show all candidates at once and ask for a ranking — one LLM call."""
    # Format candidates as a numbered list
    doc_list = "\n\n".join(
        f"[ID: {c['id']}] {c['title']}\n{c['text'][:200]}..."
        for c in candidates
    )

    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Rank the following documents by relevance to the query.
Return the document IDs in order from most to least relevant.

Query: {query}

Documents:
{doc_list}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RankedList,
            temperature=0.0,
        ),
    )
    result = RankedList.model_validate_json(response.text)

    # Re-order candidates according to the ranked list
    id_to_candidate = {c["id"]: c for c in candidates}
    reranked = []
    for rank, doc_id in enumerate(result.ranked_ids):
        if doc_id in id_to_candidate:
            reranked.append({**id_to_candidate[doc_id], "listwise_rank": rank + 1})
    # Append any not included in ranking
    ranked_ids_set = set(result.ranked_ids)
    for c in candidates:
        if c["id"] not in ranked_ids_set:
            reranked.append({**c, "listwise_rank": len(result.ranked_ids) + 1})

    print(f"LLM reasoning: {result.reasoning}")
    return reranked


print("Listwise reranking (1 LLM call vs N calls):")
listwise = listwise_rerank(query, candidates)

print(f"\n{'Rank':<8} {'Dense':>7}  Title")
print("-" * 60)
for c in listwise:
    print(f"  #{c['listwise_rank']:<5}  {c['dense_score']:.4f}  {c['title']}")

---

## 4. Score Normalisation

When combining dense scores (0–1) and LLM scores (1–5), you need normalisation  
to put them on the same scale before fusion.

In [ ]:
def min_max_normalise(values: list[float]) -> list[float]:
    """Scale values to [0, 1] range."""
    vmin, vmax = min(values), max(values)
    if vmax == vmin:
        return [1.0] * len(values)
    return [(v - vmin) / (vmax - vmin) for v in values]


def hybrid_rerank_score(
    candidates: list[dict],
    alpha: float = 0.4,   # weight for dense score (0 = all LLM, 1 = all dense)
) -> list[dict]:
    """Combine normalised dense + LLM scores into a single hybrid score."""
    dense_scores = [c["dense_score"] for c in candidates]
    llm_scores = [c["llm_score"] for c in candidates]

    norm_dense = min_max_normalise(dense_scores)
    norm_llm = min_max_normalise(llm_scores)

    result = []
    for i, c in enumerate(candidates):
        hybrid = alpha * norm_dense[i] + (1 - alpha) * norm_llm[i]
        result.append({**c, "norm_dense": norm_dense[i], "norm_llm": norm_llm[i], "hybrid_score": hybrid})

    return sorted(result, key=lambda x: x["hybrid_score"], reverse=True)


# Apply hybrid scoring to the pointwise reranked candidates
hybrid_scored = hybrid_rerank_score(reranked)

print("Hybrid score (40% dense + 60% LLM):")
print(f"{'Rank':<6} {'Dense':>7} {'LLM':>6} {'Hybrid':>8}  Title")
print("-" * 70)
for i, c in enumerate(hybrid_scored):
    print(
        f"  {i+1}    {c['norm_dense']:.4f}  {c['llm_score']}/5   {c['hybrid_score']:.4f}  "
        f"{c['title'][:40]}"
    )

---

## 5. Threshold Filtering

Even reranked results can include irrelevant documents. A relevance threshold  
ensures we only pass high-quality context to the LLM.

In [ ]:
def filter_by_threshold(
    candidates: list[dict],
    min_llm_score: int = 3,
    max_results: int = 3,
) -> list[dict]:
    """Keep only candidates above the relevance threshold."""
    filtered = [c for c in candidates if c.get("llm_score", 5) >= min_llm_score]
    return filtered[:max_results]


filtered = filter_by_threshold(reranked, min_llm_score=3, max_results=3)

print(f"After threshold filter (min_llm_score=3): {len(filtered)} of {len(reranked)} kept")
for i, c in enumerate(filtered):
    print(f"  {i+1}. [{c['llm_score']}/5] {c['title']}")

print("\n💡 Only high-confidence relevant docs reach the LLM — less noise, better answers")

---

## 6. The Complete Retrieve-then-Rerank Pipeline

In [ ]:
def rerank_rag(
    question: str,
    retrieve_k: int = 8,    # Stage 1: retrieve broadly
    final_k: int = 3,       # Stage 2: rerank and keep top N
    min_score: int = 3,     # Minimum LLM relevance score
    verbose: bool = True,
) -> str:
    """
    Two-stage retrieve-then-rerank RAG pipeline:
    1. Dense retrieval with wide net (retrieve_k)
    2. LLM reranking + threshold filtering
    3. Generate answer with best context
    """
    # Stage 1: Retrieve candidates
    candidates = dense_retrieve(question, top_k=retrieve_k)
    if verbose:
        print(f"Stage 1: Retrieved {len(candidates)} candidates")

    # Stage 2: Rerank
    reranked = llm_rerank(question, candidates)
    filtered = filter_by_threshold(reranked, min_llm_score=min_score, max_results=final_k)

    if verbose:
        print(f"Stage 2: Reranked → {len(filtered)} docs passed threshold")
        for i, c in enumerate(filtered):
            print(f"         {i+1}. [{c['llm_score']}/5] {c['title']}")

    if not filtered:
        return "I couldn't find sufficiently relevant information to answer this question."

    # Stage 3: Generate
    context = "\n\n".join(
        f"[{c['title']}]\n{c['text']}" for c in filtered
    )
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer the question using ONLY the provided research abstracts.
Cite the paper title when making claims.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


# Run the full pipeline
answer = rerank_rag(query)
print(f"\nAnswer:\n{answer}")

---

## 7. Head-to-Head: Dense-only vs Reranked RAG

In [ ]:
def dense_only_rag(question: str, top_k: int = 3) -> str:
    """Simple dense-retrieve-then-generate, no reranking."""
    candidates = dense_retrieve(question, top_k=top_k)
    context = "\n\n".join(
        f"[{c['title']}]\n{c['text']}" for c in candidates
    )
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""Answer using ONLY the provided research abstracts. Cite titles.

<context>
{context}
</context>

Question: {question}""",
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text.strip()


test_questions = [
    {
        "q": "How do you adapt a large pre-trained model without retraining all weights?",
        "best_doc": "LoRA",
    },
    {
        "q": "How can retrieval improve language models on knowledge-intensive tasks?",
        "best_doc": "RAG paper",
    },
    {
        "q": "What technique makes large language models better at arithmetic reasoning?",
        "best_doc": "Chain-of-Thought",
    },
]

print("HEAD-TO-HEAD: Dense-only vs Reranked RAG\n")
print("=" * 70)

for item in test_questions:
    q = item["q"]
    print(f"\n❓ {q}")
    print(f"   Expected best source: {item['best_doc']}")

    # Dense-only: what's the top retrieved doc?
    dense_top = dense_retrieve(q, top_k=3)
    print(f"\n   Dense-only top-3: {[c['title'][:30] for c in dense_top]}")

    # Reranked: what passes the threshold?
    reranked_docs = llm_rerank(q, dense_retrieve(q, top_k=6))
    filtered_docs = filter_by_threshold(reranked_docs, min_llm_score=3, max_results=3)
    print(f"   Reranked top-3:  {[c['title'][:30] for c in filtered_docs]}")
    print("-" * 70)

---

## 8. Performance Trade-offs

Reranking improves quality but adds latency. Here's how to think about the trade-off:

In [ ]:
import time

test_query = "What are scaling laws for neural language models?"

# Time dense-only
t0 = time.time()
dense_result = dense_only_rag(test_query, top_k=3)
dense_time = time.time() - t0

# Time reranked (with a smaller candidate pool to keep demo fast)
t0 = time.time()
candidates = dense_retrieve(test_query, top_k=5)
reranked_candidates = llm_rerank(test_query, candidates)
rerank_time = time.time() - t0

print("Latency comparison:")
print(f"  Dense-only retrieval + generation: {dense_time:.1f}s")
print(f"  Reranking phase only:              {rerank_time:.1f}s")
print(f"  Reranking overhead:                ~{rerank_time:.0f}s per {len(candidates)} candidates")

print("""
Trade-off summary:
┌──────────────────┬────────────┬──────────────┬────────────────────────┐
│ Approach         │ Latency    │ Precision    │ Best For               │
├──────────────────┼────────────┼──────────────┼────────────────────────┤
│ Dense-only       │ Fast       │ Good         │ High-traffic, low-risk │
│ LLM reranking    │ Slower     │ Better       │ High-stakes answers    │
│ Dedicated CE*    │ Moderate   │ Best         │ Production at scale    │
└──────────────────┴────────────┴──────────────┴────────────────────────┘
*CE = cross-encoder model (e.g. Cohere Rerank, ms-marco-MiniLM)
""")

---

## 🏋️ Exercise 1: Query-Specific Scoring Criteria

The generic 1–5 prompt works, but you can improve reranking by tailoring  
the scoring rubric to the query type.

In [ ]:
# Exercise 1: Domain-adapted reranker
# The default scoring prompt is domain-agnostic.
# Your task: write a reranker specialised for academic paper retrieval.
#
# Scoring criteria should consider:
#   - Direct methodological relevance (does the paper propose the method asked about?)
#   - Specificity (a paper about the exact technique scores higher than adjacent topics)
#   - Evidence quality (does the abstract report experimental results?)
#
# TODO: Modify the prompt in llm_score_pair() to include these criteria
# Then compare scores on these two queries:
#   1. "What training efficiency methods exist for large language models?"
#   2. "How does chain-of-thought improve complex reasoning?"

def academic_score_pair(query: str, document: str, title: str) -> RelevanceScore:
    """Academic-paper-specialised relevance scoring."""
    # TODO: Write a better prompt that judges papers more accurately
    # Consider: Does this paper DIRECTLY address the query? Is it the primary source?
    response = client.models.generate_content(
        model=GEN_MODEL,
        contents=f"""# TODO: Your academic reranking prompt here
Query: {query}
Paper title: {title}
Abstract: {document}""",
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=RelevanceScore,
            temperature=0.0,
        ),
    )
    return RelevanceScore.model_validate_json(response.text)


# TODO: Test your academic reranker on the two queries above
# Compare with the generic llm_score_pair results

---

## 🏋️ Exercise 2: Adaptive Retrieve-K

Currently `retrieve_k` is fixed. But simple queries need fewer candidates;  
ambiguous queries benefit from a wider net.

In [ ]:
# Exercise 2: Adaptive retrieve_k
# The idea: use query complexity to decide how many candidates to retrieve
#
# Simple query ("What is BERT?")  → retrieve_k=3  (answer is obvious)
# Complex query ("Compare training efficiency techniques for LLMs") → retrieve_k=8
#
# TODO:
# 1. Write a classify_query_complexity(query) function that returns "simple" | "moderate" | "complex"
# 2. Map each level to a retrieve_k value (3, 5, 8)
# 3. Modify rerank_rag() to use the adaptive retrieve_k
# 4. Test on 3 queries of varying complexity

class QueryComplexity(BaseModel):
    complexity: str   # "simple", "moderate", or "complex"
    reason: str


def classify_query_complexity(query: str) -> QueryComplexity:
    # TODO: Implement this
    pass


def adaptive_rerank_rag(question: str, min_score: int = 3) -> str:
    # TODO: Call classify_query_complexity, pick retrieve_k, call rerank_rag
    pass


# TODO: Test here
test_queries = [
    "What is BERT?",
    "How does self-attention work in transformers?",
    "Compare and contrast different approaches to reducing transformer memory complexity",
]

---

## Key Takeaways

| Concept | Detail |
|---------|--------|
| Two-stage retrieval | Fast bi-encoder retrieval → precise cross-encoder reranking |
| Pointwise reranking | Score each doc independently — accurate but N LLM calls |
| Listwise reranking | Score all docs at once — 1 LLM call, slightly less precise |
| Score normalisation | Min-max scale before combining dense + LLM scores |
| Threshold filtering | Drop low-relevance docs before LLM generation |
| Latency trade-off | Each reranked candidate = 1 extra LLM call — use carefully |

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Cohere Rerank API](https://docs.cohere.com/docs/reranking) | Docs | Production cross-encoder reranking service |
| [ms-marco-MiniLM](https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2) | Model | Fast open-source cross-encoder |
| [Reranking in RAG](https://www.pinecone.io/learn/series/rag/rerankers/) | Blog | Pinecone's guide to rerankers |

---

**Next up:** [04_Context_Compression_and_Filtering.ipynb](./04_Context_Compression_and_Filtering.ipynb)